In [ ]:

import os
import re
import csv
import random
import shutil
from datetime import datetime


In [ ]:
RQ1_ROOT = "../datasets/rq1_datasets"
EVENTA_ROOT = "../datasets/eventA"

RQ1_SOURCES = {
    "factory_method": {
        "true":  "../datasets/Factory method/name",
        "false": "../datasets/Factory method/nonfactorymethod",
    },
    "prototype": {
        "true":  "../datasets/Prototype/name",
        "false": "../datasets/Prototype/nonprototype",
    },
    "singleton": {
        "true":  "../datasets/Singleton/name",
        "false": "../datasets/Singleton/nonsingleton",
    },
}

def _natural_key(name):
    """Sort 'foo (2).java' before 'foo (10).java'."""
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", name)]

def build_eventA_dataset(pattern, n_total=15, seed=None):
    """Generate Event A using only files excluded from RQ1."""

    if pattern not in RQ1_SOURCES:
        raise ValueError(f"Unknown pattern: {pattern}")

    # Original source folders
    src_true = RQ1_SOURCES[pattern]["true"]
    src_false = RQ1_SOURCES[pattern]["false"]

    # Files already used in RQ1
    rq1_path = os.path.join(RQ1_ROOT, pattern)

    rq1_true = set(os.listdir(os.path.join(rq1_path, "pattern")))
    rq1_false = set(os.listdir(os.path.join(rq1_path, "non-pattern")))

    # Only use files NOT in RQ1
    true_files = sorted(
        (
            f for f in os.listdir(src_true)
            if f.endswith(".java") and f not in rq1_true
        ),
        key=_natural_key
    )

    false_files = sorted(
        (
            f for f in os.listdir(src_false)
            if f.endswith(".java") and f not in rq1_false
        ),
        key=_natural_key
    )

    # Target 7 positive + 8 negative for 15 samples
    n_true = min(n_total // 2, len(true_files))
    n_false = min(n_total - n_total // 2, len(false_files))

    # Fill shortages from either available class
    remaining = n_total - n_true - n_false

    extra_true = min(remaining, len(true_files) - n_true)
    n_true += extra_true
    remaining -= extra_true

    extra_false = min(remaining, len(false_files) - n_false)
    n_false += extra_false
    remaining -= extra_false

    if remaining:
        print(f"Warning: {pattern} is short {remaining} samples.")

    # Select files
    if seed is None:
        sel_true = true_files[:n_true]
        sel_false = false_files[:n_false]
        rule = "natural sorted order"
    else:
        rng = random.Random(seed)
        sel_true = rng.sample(true_files, n_true)
        sel_false = rng.sample(false_files, n_false)
        rule = f"random seed={seed}"

    # Create output
    target = os.path.join(EVENTA_ROOT, pattern)
    dst_true = os.path.join(target, "pattern")
    dst_false = os.path.join(target, "non-pattern")

    if os.path.exists(target):
        shutil.rmtree(target)

    os.makedirs(dst_true)
    os.makedirs(dst_false)

    for f in sel_true:
        shutil.copy2(
            os.path.join(src_true, f),
            os.path.join(dst_true, f)
        )

    for f in sel_false:
        shutil.copy2(
            os.path.join(src_false, f),
            os.path.join(dst_false, f)
        )

    # Record selection
    with open(
        os.path.join(target, "selection.csv"),
        "w",
        newline="",
        encoding="utf-8"
    ) as fh:
        writer = csv.writer(fh)
        writer.writerow(["class", "label", "filename", "source_folder"])

        writer.writerows(
            [["pattern", 1, f, src_true] for f in sel_true]
        )
        writer.writerows(
            [["non-pattern", 0, f, src_false] for f in sel_false]
        )

        writer.writerow([])
        writer.writerow([f"selection rule: {rule}"])

    print(
        f"{pattern}: {n_true} true + {n_false} false "
        f"= {n_true + n_false} files"
    )

    return {
        "pattern": pattern,
        "n_true": n_true,
        "n_false": n_false,
        "folder": target
    }


def build_all_eventA(n_total=15, seed=None):
    results = [
        build_eventA_dataset(p, n_total=n_total, seed=seed)
        for p in RQ1_SOURCES
    ]

    return results


In [ ]:
build_all_eventA(n_total=15)